# Однофакторный анализ: локальный рецидив рака ротоглотки

Исходная таблица — `w4/analytic_table.csv` (результат недели 4, признаки в исходных единицах), файл не изменяется.

Однофакторный анализ опирается на исход, поэтому выполняется только по обучающей выборке недели 5 (`w5/train.csv`, 105 пациентов).
Тестовая выборка (`w5/test.csv`, 35 пациентов) в анализе не участвует и остаётся для итоговой оценки качества модели;
из файлов недели 5 берутся только идентификаторы пациентов.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

In [2]:
analytic = pd.read_csv("../w4/analytic_table.csv")
target = "Target_local_recurrence"

train_ids = pd.read_csv("../w5/train.csv", usecols=["Patient_ID"])["Patient_ID"]
test_ids = pd.read_csv("../w5/test.csv", usecols=["Patient_ID"])["Patient_ID"]
assert set(train_ids).isdisjoint(test_ids)
assert set(train_ids) | set(test_ids) == set(analytic["Patient_ID"])

data = analytic[analytic["Patient_ID"].isin(train_ids)].reset_index(drop=True)  # тестовая выборка не используется
assert len(data) == len(train_ids)

pd.DataFrame({"пациентов": [len(data), len(test_ids)],
              "исход 1": [int(data[target].sum()), int(analytic.loc[analytic["Patient_ID"].isin(test_ids), target].sum())]},
             index=["обучение (анализ)", "тест (не используется)"])

,пациентов,исход 1
обучение (анализ),105,9
тест (не используется),35,3


In [3]:
# признак: группа сравнения
binary = {"Gender": "Female", "Induction_Chemotherapy": "N", "Concurrent_chemotherapy": "N"}
nominal = {"Race": "White", "Tumor_side": "L", "Tumor_subsite": "BOT", "Smoking_status_at_diagnosis": "Never"}

# признак: категории по порядку; группа сравнения — самая многочисленная категория в обучающей выборке
ordinal = {
    "T_category": [1, 2, 3, 4],
    "N_category": ["0", "1", "2a", "2b", "2c", "3"],
    "AJCC_Stage": ["I", "II", "III", "IV"],
    "Pathological_grade": ["I", "II", "II-III", "III", "IV"],
}
ordinal_reference = {"T_category": 2, "N_category": "2b", "AJCC_Stage": "IV", "Pathological_grade": "III"}

quant = [col for col in data.columns if col not in ["Patient_ID", target, *binary, *nominal, *ordinal]]

for col, reference in ordinal_reference.items():
    assert data[col].value_counts().idxmax() == reference, col

len(binary), len(nominal), len(ordinal), len(quant)

(3, 4, 4, 20)

## Меры связи для категориальных признаков

In [4]:
def ratio_measures(a, b, c, d):
    """a, b — исход 1 и исход 0 в группе; c, d — в группе сравнения."""
    rd = a / (a + b) - c / (c + d)
    rr = a / (a + b) / (c / (c + d)) if c > 0 else None
    odds_ratio = a * d / (b * c) if b * c > 0 else None
    rr_ci = or_ci = None
    if min(a, b, c, d) > 0:
        z = 1.96
        se_rr = np.sqrt(1 / a - 1 / (a + b) + 1 / c - 1 / (c + d))
        se_or = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
        rr_ci = np.exp(np.log(rr) + np.array([-z, z]) * se_rr)
        or_ci = np.exp(np.log(odds_ratio) + np.array([-z, z]) * se_or)
    return rd, rr, rr_ci, odds_ratio, or_ci


def show(value):
    return "не вычисляется" if value is None else f"{value:.2f}"


def show_ci(ci):
    return "не вычисляется" if ci is None else f"{ci[0]:.2f}–{ci[1]:.2f}"


def note(a, b, c, total):
    notes = []
    if c == 0:
        notes.append("в группе сравнения нет исходов")
    if a == 0:
        notes.append("в группе нет исходов")
    if b == 0:
        notes.append("в группе нет пациентов без исхода")
    if total < 10:
        notes.append("меньше 10 пациентов")
    return ", ".join(notes)


MISSING = "нет данных"


def with_missing(col):
    """Значения признака, пропуск — отдельная группа «нет данных»."""
    return data[col].astype(object).where(data[col].notna(), MISSING)


def univariate(col, order, reference):
    values = with_missing(col)
    if (values == MISSING).any():
        order = [*order, MISSING]
    assert set(values) <= set(order), col
    counts = pd.crosstab(values, data[target]).reindex(order, fill_value=0)

    table = pd.DataFrame({"Исход 1": counts[1], "Исход 0": counts[0]})
    table["Всего"] = table["Исход 1"] + table["Исход 0"]
    table["Риск"] = (table["Исход 1"] / table["Всего"]).round(3)

    c, d = table.loc[reference, ["Исход 1", "Исход 0"]]
    measures = {}
    for group, (a, b, total) in table[["Исход 1", "Исход 0", "Всего"]].iterrows():
        if group == reference:
            measures[group] = ["—"] * 5 + ["группа сравнения" + (", меньше 10 пациентов" if total < 10 else "")]
            continue
        rd, rr, rr_ci, odds_ratio, or_ci = ratio_measures(a, b, c, d)
        measures[group] = [f"{rd:+.3f}", show(rr), show_ci(rr_ci), show(odds_ratio), show_ci(or_ci), note(a, b, c, total)]
    columns = ["RD", "RR", "RR 95% ДИ", "OR", "OR 95% ДИ", "Примечание"]
    table = table.join(pd.DataFrame.from_dict(measures, orient="index", columns=columns))

    overall = table[["Исход 1", "Исход 0", "Всего"]].sum()
    table.loc["Всего"] = [*overall, round(overall["Исход 1"] / overall["Всего"], 3)] + [""] * len(columns)
    table.index = table.index.astype(str)
    table.index.name = col

    print(f"{col}: группа сравнения — {reference}")
    if (values == MISSING).any():
        print(f"пропуск признака у {(values == MISSING).sum()} пациентов — отдельная группа «{MISSING}»")
    display(table)

## Бинарные признаки

RD — разность рисков, RR — отношение рисков, OR — отношение шансов группы относительно группы сравнения, 95% ДИ — доверительный интервал Вальда.
Пациенты с пропуском категориального признака составляют отдельную группу «нет данных» — и в таблице сопряжённости, и при расчёте V Крамера.

In [5]:
for col, reference in binary.items():
    univariate(col, [reference] + [g for g in data[col].dropna().unique() if g != reference], reference)

Gender: группа сравнения — Female


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Gender,,,,,,,,,,
Female,0,13,13,0.000,—,—,—,—,—,группа сравнения
Male,9,83,92,0.098,+0.098,не вычисляется,не вычисляется,не вычисляется,не вычисляется,в группе сравнения нет исходов
Всего,9,96,105,0.086,,,,,,


Induction_Chemotherapy: группа сравнения — N


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Induction_Chemotherapy,,,,,,,,,,
N,5,61,66,0.076,—,—,—,—,—,группа сравнения
Y,4,35,39,0.103,+0.027,1.35,0.39–4.74,1.39,0.35–5.54,
Всего,9,96,105,0.086,,,,,,


Concurrent_chemotherapy: группа сравнения — N


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Concurrent_chemotherapy,,,,,,,,,,
N,1,30,31,0.032,—,—,—,—,—,группа сравнения
Y,8,66,74,0.108,+0.076,3.35,0.44–25.68,3.64,0.44–30.39,
Всего,9,96,105,0.086,,,,,,


## Номинальные признаки

In [6]:
for col, reference in nominal.items():
    others = data[col].value_counts().drop(reference).index.tolist()  # по убыванию числа пациентов
    univariate(col, [reference] + others, reference)

Race: группа сравнения — White


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Race,,,,,,,,,,
White,8,92,100,0.080,—,—,—,—,—,группа сравнения
Black,0,3,3,0.000,-0.080,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Hispanic,1,1,2,0.500,+0.420,6.25,1.34–29.07,11.50,0.66–201.71,меньше 10 пациентов
Всего,9,96,105,0.086,,,,,,


Tumor_side: группа сравнения — L


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Tumor_side,,,,,,,,,,
L,5,52,57,0.088,—,—,—,—,—,группа сравнения
R,4,44,48,0.083,-0.004,0.95,0.27–3.34,0.95,0.24–3.74,
Всего,9,96,105,0.086,,,,,,


Tumor_subsite: группа сравнения — BOT


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Tumor_subsite,,,,,,,,,,
BOT,2,46,48,0.042,—,—,—,—,—,группа сравнения
Tonsil,5,35,40,0.125,+0.083,3.00,0.61–14.64,3.29,0.60–17.95,
Other,1,6,7,0.143,+0.101,3.43,0.36–33.05,3.83,0.30–48.94,меньше 10 пациентов
GPS,0,7,7,0.000,-0.042,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Soft_palate,1,1,2,0.500,+0.458,12.00,1.73–83.46,23.00,1.02–516.96,меньше 10 пациентов
Pharyngeal_wall,0,1,1,0.000,-0.042,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Всего,9,96,105,0.086,,,,,,


Smoking_status_at_diagnosis: группа сравнения — Never


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Smoking_status_at_diagnosis,,,,,,,,,,
Never,2,31,33,0.061,—,—,—,—,—,группа сравнения
Former,5,35,40,0.125,+0.064,2.06,0.43–9.95,2.21,0.40–12.24,
Current,2,30,32,0.062,+0.002,1.03,0.15–6.89,1.03,0.14–7.82,
Всего,9,96,105,0.086,,,,,,


## Порядковые признаки

In [7]:
for col, order in ordinal.items():
    univariate(col, order, ordinal_reference[col])

T_category: группа сравнения — 2


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
T_category,,,,,,,,,,
1,0,26,26,0.000,-0.146,0.00,не вычисляется,0.00,не вычисляется,в группе нет исходов
2,6,35,41,0.146,—,—,—,—,—,группа сравнения
3,2,19,21,0.095,-0.051,0.65,0.14–2.95,0.61,0.11–3.34,
4,1,16,17,0.059,-0.088,0.40,0.05–3.09,0.36,0.04–3.28,
Всего,9,96,105,0.086,,,,,,


N_category: группа сравнения — 2b


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
N_category,,,,,,,,,,
0,0,7,7,0.000,-0.074,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
1,0,14,14,0.000,-0.074,0.00,не вычисляется,0.00,не вычисляется,в группе нет исходов
2a,0,5,5,0.000,-0.074,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
2b,4,50,54,0.074,—,—,—,—,—,группа сравнения
2c,5,18,23,0.217,+0.143,2.93,0.87–9.95,3.47,0.84–14.38,
3,0,2,2,0.000,-0.074,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Всего,9,96,105,0.086,,,,,,


AJCC_Stage: группа сравнения — IV


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
AJCC_Stage,,,,,,,,,,
I,0,1,1,0.000,-0.103,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
II,0,3,3,0.000,-0.103,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
III,0,14,14,0.000,-0.103,0.00,не вычисляется,0.00,не вычисляется,в группе нет исходов
IV,9,78,87,0.103,—,—,—,—,—,группа сравнения
Всего,9,96,105,0.086,,,,,,


Pathological_grade: группа сравнения — III
пропуск признака у 13 пациентов — отдельная группа «нет данных»


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Pathological_grade,,,,,,,,,,
I,0,1,1,0.000,-0.083,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
II,4,36,40,0.100,+0.017,1.20,0.32–4.50,1.22,0.29–5.23,
II-III,0,2,2,0.000,-0.083,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
III,4,44,48,0.083,—,—,—,—,—,группа сравнения
IV,0,1,1,0.000,-0.083,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
нет данных,1,12,13,0.077,-0.006,0.92,0.11–7.57,0.92,0.09–8.98,
Всего,9,96,105,0.086,,,,,,


## Коэффициент V Крамера

In [8]:
def cramers_v(col):
    counts = pd.crosstab(with_missing(col), data[target]).to_numpy()  # «нет данных» — отдельная группа
    n = counts.sum()
    expected = counts.sum(axis=1, keepdims=True) * counts.sum(axis=0, keepdims=True) / n
    chi2 = ((counts - expected) ** 2 / expected).sum()
    return n, len(counts), np.sqrt(chi2 / (n * (min(counts.shape) - 1))), (expected < 5).mean()


kinds = {**dict.fromkeys(binary, "бинарный"), **dict.fromkeys(nominal, "номинальный"),
         **dict.fromkeys(ordinal, "порядковый")}
cramer = pd.DataFrame([cramers_v(col) for col in kinds], index=list(kinds),
                      columns=["пациентов", "групп", "V Крамера", "доля ячеек с ожидаемой частотой < 5"])
cramer.insert(0, "тип", pd.Series(kinds))
cramer.round(3).sort_values("V Крамера", ascending=False)

,тип,пациентов,групп,V Крамера,доля ячеек с ожидаемой частотой < 5
N_category,порядковый,105,6,0.273,0.667
Tumor_subsite,номинальный,105,6,0.265,0.667
Race,номинальный,105,3,0.212,0.667
T_category,порядковый,105,4,0.208,0.500
AJCC_Stage,порядковый,105,4,0.139,0.625
Concurrent_chemotherapy,бинарный,105,2,0.124,0.250
Gender,бинарный,105,2,0.115,0.250
Smoking_status_at_diagnosis,номинальный,105,3,0.110,0.500
Pathological_grade,порядковый,105,6,0.069,0.750
Induction_Chemotherapy,бинарный,105,2,0.046,0.250


## Количественные признаки

OR при увеличении признака на межквартильный размах (МКР = Q3 − Q1) — из однофакторной логистической регрессии по признаку в исходных единицах.
Пациенты с пропуском количественного признака в его модель не входят.

Силу связи показывает само OR, а ДИ — точность его оценки. OR меньше 1 сравнивается с остальными по 1/OR, поэтому таблица упорядочена по столбцу «сила связи» = max(OR, 1/OR).

In [9]:
def logistic(x, y):
    """Логистическая регрессия y ~ 1 + x методом Ньютона, возвращает наклон и его стандартную ошибку."""
    X = np.column_stack([np.ones_like(x), x])
    beta = np.zeros(2)
    for _ in range(100):
        p = 1 / (1 + np.exp(-X @ beta))
        hessian = X.T @ (X * (p * (1 - p))[:, None])
        step = np.linalg.solve(hessian, X.T @ (y - p))
        beta += step
        if np.abs(step).max() < 1e-10:
            break
    p = 1 / (1 + np.exp(-X @ beta))
    covariance = np.linalg.inv(X.T @ (X * (p * (1 - p))[:, None]))
    return beta[1], np.sqrt(covariance[1, 1])

In [10]:
units = {
    "Age_at_diagnosis": "лет",
    "Smoking_Pack-Years": "пачко-лет",
    "Total_prescribed_Radiation_treatment_dose": "Гр",
    "Dose_per_fraction": "Гр за фракцию",
    "GTVp_shape_MeshVolume": "мм³",
    "GTVp_shape_Sphericity": "безразмерный",
    "GTVp_shape_Elongation": "безразмерный",
    "GTVp_shape_Flatness": "безразмерный",
    "GTVp_firstorder_10Percentile": "HU",
    "GTVp_firstorder_90Percentile": "HU",
    "GTVp_firstorder_Entropy": "бит",
    "GTVp_firstorder_Kurtosis": "безразмерный",
    "GTVp_firstorder_Maximum": "HU",
    "GTVp_firstorder_Minimum": "HU",
    "GTVp_glcm_Correlation": "безразмерный",
    "GTVp_glcm_InverseVariance": "безразмерный",
    "GTVp_gldm_DependenceNonUniformityNormalized": "безразмерный",
    "GTVp_glszm_SizeZoneNonUniformityNormalized": "безразмерный",
    "GTVp_ngtdm_Busyness": "усл. ед.",
    "GTVp_ngtdm_Contrast": "усл. ед.",
}

def show_number(value):
    return np.format_float_positional(value, precision=4, unique=False, fractional=False, trim="-")


rows = {}
for col in quant:
    known = data[col].notna()
    x, y = data.loc[known, col].to_numpy(float), data.loc[known, target].to_numpy(float)
    q1, q3 = np.quantile(x, [0.25, 0.75])
    slope, se = logistic((x - np.median(x)) / (q3 - q1), y)  # наклон на один МКР
    ci = np.exp(slope + np.array([-1.96, 1.96]) * se)
    rows[col] = [known.sum(), int(y.sum()), units[col], show_number(q1), show_number(q3), show_number(q3 - q1),
                 round(np.exp(slope), 2), f"{ci[0]:.2f}–{ci[1]:.2f}", round(np.exp(abs(slope)), 2),
                 "да" if ci[0] > 1 or ci[1] < 1 else "нет"]

(pd.DataFrame.from_dict(rows, orient="index",
                        columns=["пациентов", "исход 1", "единицы", "Q1", "Q3", "МКР", "OR на МКР", "OR 95% ДИ",
                                 "сила связи", "ДИ не включает 1"])
   .sort_values("сила связи", ascending=False))

,пациентов,исход 1,единицы,Q1,Q3,МКР,OR на МКР,OR 95% ДИ,сила связи,ДИ не включает 1
Total_prescribed_Radiation_treatment_dose,105,9,Гр,66,70,4,2.33,0.56–9.68,2.33,нет
Smoking_Pack-Years,96,9,пачко-лет,0,30,30,2.31,1.03–5.19,2.31,да
Age_at_diagnosis,105,9,лет,54,66,12,2.28,0.91–5.70,2.28,нет
GTVp_firstorder_Minimum,100,9,HU,-755,-299.8,455.2,0.44,0.14–1.38,2.27,нет
GTVp_ngtdm_Busyness,104,9,усл. ед.,0.48,2.16,1.68,0.53,0.17–1.63,1.89,нет
GTVp_ngtdm_Contrast,104,9,усл. ед.,0.002749,0.01012,0.007374,0.55,0.18–1.65,1.82,нет
GTVp_firstorder_Kurtosis,104,9,безразмерный,25.79,71.52,45.73,1.80,0.76–4.27,1.80,нет
GTVp_glcm_InverseVariance,104,9,безразмерный,0.411,0.4547,0.04375,1.77,0.67–4.69,1.77,нет
Dose_per_fraction,105,9,Гр за фракцию,2.121,2.2,0.07879,1.45,0.78–2.70,1.45,нет
GTVp_firstorder_Entropy,104,9,бит,1.722,2.456,0.7337,1.43,0.70–2.91,1.43,нет


## Выводы

Анализ выполнен по обучающей выборке: рецидив был у 9 из 105 пациентов (риск 0.086), меры неточны. Тестовая выборка (35 пациентов, 3 рецидива) не использовалась.

**Сильнее всего связаны с исходом.** Силу связи оценивали по самой мере: V Крамера для категориальных признаков и OR на МКР для количественных, причём OR меньше 1 сравнивали с остальными по 1/OR; ДИ показывает только точность оценки.
Среди категориальных признаков — категория N (V 0.273) и подлокализация (V 0.265): риск 0.217 при N2c против 0.074 при N2b (RR 2.93, ДИ 0.87–9.95), при N0, N1 и N2a рецидивов нет (0 из 26); риск 0.125 при опухоли миндалины против 0.042 в корне языка (RR 3.00), при опухоли мягкого нёба рецидив у 1 из 2 (RR 12.00). Далее раса (V 0.212, за счёт 1 рецидива у 2 пациентов латиноамериканского происхождения) и категория T (V 0.208): при T1 рецидивов нет (0 из 26). Все 9 рецидивов — при IV стадии (0 из 18 при I–III).
Среди количественных признаков наибольшая сила связи у полной дозы (OR 2.33 на 4 Гр), пачко-лет (OR 2.31 на 30 пачко-лет), возраста (OR 2.28 на 12 лет) и минимальной плотности опухоли (OR 0.44 на 455 HU, то есть 1/OR 2.27: чем ниже минимальная плотность, тем выше риск). Связь минимальной плотности по силе почти такая же, как у пачко-лет, и сильнее, чем у максимальной плотности (OR 1.40). Далее неоднородность текстуры (busyness, 1/OR 1.89; contrast, 1/OR 1.82). Только у пачко-лет 95% ДИ не включает 1 (1.03–5.19), но это говорит о точности оценки, а не о том, что связь сильнее.

**Согласие с известным о болезни.** Курение, распространённость по N и стадия, локализация вне корня языка ухудшают локальный контроль, связи с этим согласуются. Связь с категорией T немонотонна (наибольший риск при T2, 0.146), что скорее объясняется малым числом исходов. Очень низкая минимальная плотность (около −750 HU) соответствует воздуху или некрозу в контуре опухоли; некроз и гипоксия связаны с радиорезистентностью, но значения могут отражать и прилегание опухоли к воздухоносным путям, поэтому объяснение предположительное.

**Связи, которые может объяснять другой признак.** Одновременная химиотерапия (RR 3.35, 0.44–25.68) и полная доза назначаются при распространённой опухоли, их связь может объясняться стадией и категорией N. Пачко-годы и статус курения отражают одно воздействие, при этом риск у бывших курильщиков выше, чем у курящих (0.125 и 0.062). Минимальная плотность и текстурные признаки вычислены по одному контуру и могут быть связаны с размером и локализацией опухоли.

**Пропуски.** Степень дифференцировки неизвестна у 13 пациентов; они составляют отдельную группу «нет данных» (риск 0.077, близок к среднему) и учтены в таблице и при расчёте V Крамера (0.069, 6 групп). Пациенты с пропуском количественного признака в его модель не входят (до 9 пациентов у пачко-лет).

**Малое число наблюдений.** Мера для мягкого нёба получена по 2 пациентам (ДИ OR 1.02–516.96), ещё 14 групп меньше 10 пациентов отмечены в таблицах, в группах без исходов RR и OR равны 0. У женщин (группа сравнения по полу) исходов нет, поэтому RR и OR для пола не вычисляются. У всех категориальных признаков ожидаемая частота меньше 5 в 25–75% ячеек.